# D222 long-document continuation
Use the Google account holding the completed original D222 run. Select T4 and run the single code cell. Original outputs are preserved.
Requires the existing original input ZIP, completed Phase4 best.pt and complete.json. Reserve at least 4 GB additional Drive space; exact peak estimate prints during preflight.
Eight 256-token windows, small FP32 BiGRU residual across ordered chunks, two chunks per D222 backbone batch. Fixed original holdout and data/parent/tokenizer provenance are enforced.
Fresh optimizer; this is a new architecture continuation, not exact optimizer resume. Three-hour wall-time training budget includes evaluations; data preparation and final export add time. Deadline cutoff reserves 75 minutes.
Source has been parsed locally; actual T4 execution and score improvement are unverified. Complete authentication, then confirm increasing training steps before leaving. This additional LoRMA variant does not replace any of the six distinct required techniques.


In [ ]:
# D222 LONG-DOCUMENT CONTINUATION: run after your original D222 finishes.
# Existing Drive input/checkpoint are reused. Original results remain untouched.
import os,sys,time,json,hashlib,shutil,subprocess,importlib.util,gc,zipfile,base64,io
from pathlib import Path
for proc in Path('/proc').iterdir():
 if not proc.name.isdigit():continue
 try:args=proc.joinpath('cmdline').read_bytes().split(b'\0')
 except (FileNotFoundError,PermissionError,ProcessLookupError):continue
 if b'phase4_cjpe.runtime' in args and b'--worker' in args:raise RuntimeError('A Phase4 worker is already running here. Wait for it to finish. PID '+proc.name)
for module,package in [('ijson','ijson>=3.2,<4'),('huggingface_hub','huggingface_hub>=0.26,<2'),('sklearn','scikit-learn>=1.4,<2'),('tokenizers','tokenizers')]:
 if importlib.util.find_spec(module) is None:subprocess.check_call([sys.executable,'-m','pip','install','-q',package])
import torch
assert torch.cuda.is_available(),'Choose Runtime > Change runtime type > T4 GPU.'
print('ACTUAL GPU:',torch.cuda.get_device_name(0),flush=True)
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir():drive.mount('/content/drive')
PERSIST=Path('/content/drive/MyDrive/CS781_Phase4')
WORK=Path('/content/phase4_work');WORK.mkdir(parents=True,exist_ok=True)
assert shutil.disk_usage('/content').free>12e9,'Need 12 GB free temporary runtime disk.'
PARENT=PERSIST/'runs/01_D222_LoRMA'
assert (PARENT/'complete.json').is_file(),'Completed original D222 result missing from this Drive account. Use the account containing runs/01_D222_LoRMA.'
parent_result=json.loads((PARENT/'complete.json').read_text())
WARM=PARENT/'best.pt'
assert WARM.is_file(),'Original selected best.pt is missing. Do not upload a prediction ZIP as weights.'
INPUT=PERSIST/'inputs/Phase3_D222_Input.zip'
assert INPUT.is_file(),'Need the original Phase3_D222_Input.zip in MyDrive/CS781_Phase4/inputs.'
CODE_ROOT=Path('/content/phase4_d222_longdoc_code')
source_bytes=base64.b64decode('...')
assert hashlib.sha256(source_bytes).hexdigest()=='28faefc720376f6d9ce431a847dff8b812a5176765b00f565ebb00cf6110a041'
with zipfile.ZipFile(io.BytesIO(source_bytes)) as z:
 assert z.testzip() is None
 for name in z.namelist():
  p=Path(name);assert not p.is_absolute() and '..' not in p.parts
  dest=CODE_ROOT/p;dest.parent.mkdir(parents=True,exist_ok=True);dest.write_bytes(z.read(name))
sys.path[:]=[str(CODE_ROOT)]+[p for p in sys.path if p!=str(CODE_ROOT)]
for module in list(sys.modules):
 if module in ('phase4_cjpe','phase3_rr') or module.startswith(('phase4_cjpe.','phase3_rr.')):del sys.modules[module]
importlib.invalidate_caches()
from phase4_cjpe.common import sha,atomic_json,source_hashes,now
print('Verifying existing input and selected Phase4 checkpoint SHA...',flush=True)
assert INPUT.stat().st_size==888831449
assert sha(INPUT)=='9aee434c9b9738c90a38b5aba8bfd8a204146319e251862406b338136083ae44','Wrong original D222 input.'
assert sha(WARM)==parent_result['selected_sha256'],'Selected Phase4 checkpoint SHA mismatch.'
RUN_ROOT=PERSIST/'runs/07_D222_LongDoc_LoRMA'
RUN_ROOT.mkdir(parents=True,exist_ok=True)
DATA_ROOT=WORK/'Phase_4_CJPE'
from phase4_cjpe.data import acquire
missing=any(not (DATA_ROOT/p).is_file() for p in ('Train/train.json','Train/dev.json','Test/test.json'))
data_zip=next((p for p in (PERSIST/'inputs/Phase_4_CJPE.zip',Path('/content/Phase_4_CJPE.zip')) if p.is_file()),None)
try:DATA_ROOT=acquire(DATA_ROOT,uploaded_zip=data_zip if missing else None,download=bool(missing and not data_zip))
except Exception as exc:
 if getattr(getattr(exc,'response',None),'status_code',None) not in (401,403):raise
 from getpass import getpass
 from huggingface_hub import login
 token=getpass('Approved Hugging Face READ token (hidden): ')
 try:login(token=token,add_to_git_credential=False)
 finally:del token
 DATA_ROOT=acquire(DATA_ROOT,download=True)
CONFIG={'name':'07_D222_LongDoc_LoRMA','family':'d222','method':'lorma','experiment':'adaptation',
 'rank':8,'alpha':16,'seed':781,'max_tokens':256,'chunks_per_case':8,'region_chars':8192,
 'max_train_cases':12000,'holdout_per_class':256,'max_epochs':4,'accumulation':8,
 'backbone_lr':5e-5,'head_lr':1e-4,'training_hours':3.,'eval_every':50,'save_every':25,'patience':5,
 'teacher_hours':1.5,'kd_temperature':2.,'kd_weight':.5,'stall_minutes':45,
 'export_reserve_minutes':75,'deadline_utc':'2026-10-21T18:29:00Z','allow_after_deadline':False,
 'run_root':str(RUN_ROOT),'work_root':str(WORK),'data_root':str(DATA_ROOT),'bundle':str(INPUT),
 'warm_start':str(WARM),'document_architecture':'ordered_residual_bigru_v1'}
from phase4_cjpe.train import deadline_timestamp
assert time.time()<deadline_timestamp(CONFIG)-CONFIG['export_reserve_minutes']*60,'Training cutoff passed. Existing submission remains valid; no deadline bypass.'
atomic_json(RUN_ROOT/'continuation_plan.json',{'parent_complete_sha256':sha(PARENT/'complete.json'),
 'parent_selected_sha256':sha(WARM),'parent_dev_macro_f1':parent_result['dev_metrics']['macro_f1'],
 'new_optimizer':True,'source':source_hashes(),'created_utc':now(),
 'note':'Same own Phase1/Phase4 lineage. Four to eight document windows; residual ordered chunk encoder. Additional LoRMA variant does not replace a required distinct technique.'})
revision=DATA_ROOT/'download_revision.json'
if revision.exists():shutil.copy2(revision,RUN_ROOT/'official_download_revision.json')

# GPU preparation verifies original tokenizer and finite gradients; no optimizer update here.
from phase3_rr.archive_inputs import resolve_bundle
from phase3_rr.data import Tokenizer
from phase4_cjpe.models import make_model
bundle,input_manifest=resolve_bundle(INPUT,WORK/'weights')
assert input_manifest['family']=='d222'
tok=Tokenizer(bundle/'tokenizer.json')
sample='The Supreme Court considers the appeal under section १२३.'
ids=tok.encode(sample)
assert ids and tok.impl.decode(ids)==sample,'Original Unicode13 tokenizer round-trip failed.'
model=old=logits=loss=scaler=grads=None
try:
 model,evidence=make_model(bundle,'d222','lorma',8,16)
 old=torch.load(WARM,map_location='cpu',weights_only=False)
 assert old['evidence']['family']=='d222' and old['evidence']['method']=='lorma'
 missing_keys,unexpected=model.load_state_dict(old['model'],strict=False)
 allowed={k for k in model.state_dict() if k.startswith(('context_encoder.','context_projection.'))}
 assert not unexpected and set(missing_keys)<=allowed,(missing_keys,unexpected)
 del old;old=None
 assert bool((model.context_projection.weight==0).all()),'New residual projection must initialise to zero.'
 state_bytes=sum(t.numel()*t.element_size() for t in model.state_dict().values())
 trainable_bytes=sum(p.numel()*p.element_size() for p in model.parameters() if p.requires_grad)
 estimated_peak=int(3*state_bytes+4*trainable_bytes+300e6)
 print('DRIVE SPACE: reserve at least',round(estimated_peak/1e9,2),'GB ADDITIONAL free for best/latest checkpoint replacement. Mounted disk figures are not a Drive quota check.',flush=True)
 model=model.to('cuda').train()
 seq=[1]+ids[:30]+[2]
 with torch.autocast(device_type='cuda',dtype=torch.float16):logits=model([seq,seq[:-3]],[0.,1.],torch.device('cuda')).float()
 loss=torch.nn.functional.cross_entropy(logits,torch.tensor([0],device='cuda'))
 assert bool(torch.isfinite(loss)),'Nonfinite warm-start preflight loss.'
 scaler=torch.amp.GradScaler('cuda',init_scale=128);scaler.scale(loss).backward()
 grads=[p.grad for p in model.backbone.parameters() if p.requires_grad and p.grad is not None]
 assert grads and all(bool(torch.isfinite(g).all()) for g in grads) and any(bool(g.abs().max()>0) for g in grads),'Missing/nonfinite backbone gradients.'
 assert model.context_projection.weight.grad is not None and bool(torch.isfinite(model.context_projection.weight.grad).all()),'Missing/nonfinite new document-head gradients.'
 atomic_json(RUN_ROOT/'preflight.json',{'actual_gpu':torch.cuda.get_device_name(0),'finite_loss':float(loss.detach()),
  'parent_selected_sha256':sha(WARM),'new_parameters':list(missing_keys),'estimated_additional_drive_peak_bytes':estimated_peak,
  'drive_quota_verified':False,'optimizer_steps':0,'parameters':evidence['parameters'],'checked_utc':now()})
 print('TOKENIZER + WARM-START + GPU PREFLIGHT PASSED. Optimizer updates so far: 0.',flush=True)
finally:
 del model,old,logits,loss,scaler,grads;gc.collect();torch.cuda.empty_cache()
from phase4_cjpe.runtime import supervise
from phase4_cjpe.export import validate_zip
try:RESULT=supervise(CONFIG)
except Exception:
 failure=RUN_ROOT/'failure.json'
 if failure.exists():print(json.loads(failure.read_text()).get('traceback',''))
 raise
official_ids=json.loads((RUN_ROOT/'splits.json').read_text())['test']
checks=validate_zip(RESULT['submission'],official_ids);assert checks['sha256']==RESULT['sha256']
READY=PERSIST/'SUBMISSIONS';READY.mkdir(parents=True,exist_ok=True)
SUBMISSION=READY/Path(RESULT['submission']).name
shutil.copy2(RESULT['submission'],SUBMISSION);assert sha(SUBMISSION)==RESULT['sha256']
atomic_json(READY/(SUBMISSION.stem+'_RESULT.json'),RESULT)
print('VERIFIED SUBMISSION ZIP:',SUBMISSION,flush=True)
print('LOCAL DEV MACRO F1:',RESULT['dev_metrics']['macro_f1'],flush=True)
print('PARENT LOCAL DEV MACRO F1:',parent_result['dev_metrics']['macro_f1'],flush=True)
print('SELECTED NEW-RUN STEP:',RESULT['selected_step'],'TOTAL NEW UPDATES:',RESULT['optimizer_updates'],flush=True)
print('COMPLETED AFTER DEADLINE:',RESULT['completed_after_deadline'],flush=True)
from IPython.display import display,FileLink
display(FileLink(str(SUBMISSION)))
# A small source package preserves the exact implementation without duplicating weights.
record_path=RUN_ROOT/'EXACT_SOURCE_07_D222_LongDoc.zip'
record_path.write_bytes(source_bytes)
assert sha(record_path)==hashlib.sha256(source_bytes).hexdigest()
print('Best model, full resume, tokenizer, selection and configuration remain in:',RUN_ROOT,flush=True)
